# 05 — Stage 1: Ensemble Training

Train and cross-validate the ML super-learner ensemble for predicting third-party
military intervention.

**Inputs**: `data/interim/dd_spat_{cy}_{ud}.parquet` (25 files from notebook 04)

**Outputs**:
- `data/interim/sl_oof_{cy}_{ud}.parquet` — out-of-fold probability predictions
  for every onset dyad-year (25 files)
- `data/interim/sl_weights_{cy}_{ud}.parquet` — NNLS ensemble weights per imputation
- `results/tables/tab-tuning-by-model.tex` — component model performance

**Reference R scripts**: `14-trainModels.R`, `13-makePRcomp.R`, `16-makePirate.R`

**Pipeline per imputation dataset**:
1. Extract onset rows in Regan period; build feature matrix
2. PCA: retain components to scree-plot elbow
3. 10-fold CV, stratified by onset (leave-one-conflict-out where possible):
   - Random forest, elastic net, multinomial logit, MLP
4. NNLS super learner: minimise held-out log-loss
5. Save out-of-fold predictions and ensemble weights

**Outcome variable**: `intervention` — 0 (none), 1 (gov-biased), 2 (opp-biased)

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.optimize import nnls
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.base import clone
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import log_loss, roc_auc_score
import joblib

sys.path.insert(0, str(Path("..").resolve() / "src"))
from shadow.data.spatial import update_spatial_lags_proba, _build_W_cache, SPAT_COLS

warnings.filterwarnings("ignore")

ROOT    = Path("..").resolve()
INTERIM = ROOT / "data" / "interim"
RESULTS = ROOT / "results"
(RESULTS / "tables").mkdir(parents=True, exist_ok=True)

N_FOLDS    = 10
MAX_FP_ITER = 5      # maximum fixed-point iterations
FP_TOL     = 1e-4   # convergence: mean |Δ spat_gov| + |Δ spat_opp|
SEED       = 90210


## § 1  Feature & outcome columns

In [ ]:
# Columns that identify a row — not used as features
ID_COLS = [
    "ccode_A", "ccode_B", "year", "ddyear",
    "onset_A", "regan_period", "intervention",
]

# Spatial lag columns (excluded from PCA feature matrix; used separately)
SPAT_COLS = [
    "spat_gov", "spat_opp",
    "spat_US_G", "spat_USSR_G", "spat_US_O", "spat_USSR_O",
    "spat_US_USRG", "spat_US_USRO", "spat_USR_USG", "spat_USR_USO",
]

def get_feature_cols(df: pd.DataFrame) -> list[str]:
    """All numeric columns suitable for the PCA feature matrix."""
    exclude = set(ID_COLS)
    return [
        c for c in df.columns
        if c not in exclude and pd.api.types.is_numeric_dtype(df[c])
    ]


def prl(y_true: np.ndarray, proba: np.ndarray) -> float:
    """Proportional Reduction in Loss vs. class-frequency null."""
    null_p = np.bincount(y_true.astype(int), minlength=3) / len(y_true)
    null_p = np.clip(null_p, 1e-9, None)
    null_loss = -np.log(null_p[y_true.astype(int)]).mean()
    model_loss = log_loss(y_true, proba, labels=[0, 1, 2])
    return (null_loss - model_loss) / null_loss


## § 2  Super-learner training function

In [ ]:
def make_classifiers(seed: int) -> dict:
    """Instantiate the four component classifiers."""
    return {
        "rf":     RandomForestClassifier(
                      n_estimators=500, n_jobs=-1, random_state=seed),
        "glmnet": LogisticRegression(
                      penalty="elasticnet",
                      solver="saga", l1_ratio=0.5, C=1.0,
                      max_iter=2000, random_state=seed),
        "multinom": LogisticRegression(
                      penalty=None,
                      solver="lbfgs", max_iter=2000, random_state=seed),
        "mlp":    MLPClassifier(
                      hidden_layer_sizes=(50,), max_iter=1000,
                      early_stopping=True, random_state=seed),
    }


def _predict_proba_3class(clf, X_val: np.ndarray) -> np.ndarray:
    """Return (n, 3) probability array aligned to classes [0, 1, 2]."""
    raw = clf.predict_proba(X_val)
    out = np.zeros((len(X_val), 3))
    for j, cls in enumerate(clf.classes_):
        out[:, int(cls)] = raw[:, j]
    return out


def train_super_learner(
    X: np.ndarray,
    y: np.ndarray,
    n_components_pca: int | None = None,
    seed: int = SEED,
) -> dict:
    """
    Train a NNLS super-learner ensemble via 10-fold CV.

    Returns a dict with keys:
      'oof_proba'  : (n, 3) array of out-of-fold probabilities
      'weights'    : dict {name: float} ensemble weights (sum to 1)
      'pca'        : fitted PCA
      'scaler'     : fitted StandardScaler
      'classifiers': dict of fitted component classifiers (full data)
      'component_metrics': DataFrame with per-component CV metrics
      'n_components_pca': int
    """
    scaler = StandardScaler()
    X_sc = scaler.fit_transform(X)

    # PCA: fit on full data, choose n_components to reach 90% variance
    pca_full = PCA(random_state=seed).fit(X_sc)
    if n_components_pca is None:
        cumvar = np.cumsum(pca_full.explained_variance_ratio_)
        n_components_pca = int(np.searchsorted(cumvar, 0.90)) + 1
        n_components_pca = max(5, min(n_components_pca, X_sc.shape[1] - 1))
    pca = PCA(n_components=n_components_pca, random_state=seed).fit(X_sc)
    X_pc = pca.transform(X_sc)

    classifiers = make_classifiers(seed)
    cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)

    # Collect out-of-fold predictions for each component
    oof: dict[str, np.ndarray] = {
        name: np.zeros((len(y), 3)) for name in classifiers
    }

    for fold, (tr_idx, val_idx) in enumerate(cv.split(X_pc, y)):
        X_tr, X_val = X_pc[tr_idx], X_pc[val_idx]
        y_tr = y[tr_idx]
        for name, clf in classifiers.items():
            from sklearn.base import clone
            clf_fold = clone(clf)
            clf_fold.fit(X_tr, y_tr)
            oof[name][val_idx] = _predict_proba_3class(clf_fold, X_val)

    # NNLS stacking on one-hot labels
    names = list(classifiers.keys())
    K = len(names)
    A = np.column_stack([oof[name].reshape(-1) for name in names])
    b = np.eye(3)[y.astype(int)].reshape(-1)
    raw_w, _ = nnls(A, b)
    w_sum = raw_w.sum()
    weights = {name: float(raw_w[k] / w_sum if w_sum > 0 else 1.0 / K)
               for k, name in enumerate(names)}

    # Ensemble OOF probabilities
    oof_ensemble = sum(weights[name] * oof[name] for name in names)

    # Per-component metrics
    metrics = []
    for name in names:
        ll = log_loss(y, oof[name], labels=[0, 1, 2])
        try:
            auc = roc_auc_score(
                (y > 0).astype(int), oof[name][:, 1:].sum(axis=1)
            )
        except Exception:
            auc = np.nan
        metrics.append({"method": name, "log_loss": ll,
                        "auc": auc, "weight": weights[name]})
    ll_ens = log_loss(y, oof_ensemble, labels=[0, 1, 2])
    auc_ens = roc_auc_score(
        (y > 0).astype(int), oof_ensemble[:, 1:].sum(axis=1)
    )
    metrics.append({"method": "super_learner", "log_loss": ll_ens,
                    "auc": auc_ens, "weight": 1.0})
    metrics_df = pd.DataFrame(metrics)
    metrics_df["prl"] = prl(y, oof_ensemble)

    # Retrain on full data
    fitted_clfs = {}
    for name, clf in classifiers.items():
        clf.fit(X_pc, y)
        fitted_clfs[name] = clf

    return {
        "oof_proba":         oof_ensemble,
        "oof_per_model":     oof,
        "weights":           weights,
        "pca":               pca,
        "scaler":            scaler,
        "classifiers":       fitted_clfs,
        "component_metrics": metrics_df,
        "n_components_pca":  n_components_pca,
    }


## § 3  Main loop: train over 25 imputations

In [ ]:
all_metrics = []

for cy in range(1, 6):
    for ud in range(1, 6):
        dd = pd.read_parquet(INTERIM / f"dd_spat_{cy}_{ud}.parquet")

        # Training set: onset rows in Regan period with known intervention status
        train_base = dd[
            (dd["onset_A"] == 1) &
            (dd["regan_period"] == True) &
            (dd["intervention"].notna())
        ].copy()

        y = train_base["intervention"].astype(int).values

        # Pre-build W matrix cache (polity similarity weights) once per imputation.
        # The W matrices don't change across fixed-point iterations.
        onset_mask_full = (dd["onset_A"] == 1)
        W_cache = _build_W_cache(dd, onset_mask_full)

        print(f"── CY {cy}/5, UD {ud}/5 ── n={len(train_base):,} onset dyads, "
              f"{(y==1).sum()} gov, {(y==2).sum()} opp")

        # ── Fixed-point iteration ──────────────────────────────────────────
        # Iteration 0: train on observed spatial lags (from notebook 04).
        # Subsequent iterations: update spatial lags from OOF predictions,
        # then retrain.  Converges when the spatial lags stop changing.
        train = train_base.copy()
        result = None
        prev_spat = None

        for fp_iter in range(MAX_FP_ITER):
            feat_cols = get_feature_cols(train)
            X = train[feat_cols].fillna(0).to_numpy(dtype=float)

            result = train_super_learner(X, y, seed=SEED + cy * 10 + ud)

            # OOF predictions → update spatial lags for next iteration.
            p_gov = result["oof_proba"][:, 1]
            p_opp = result["oof_proba"][:, 2]

            # Build a full-index probability series aligned to train.
            pg_s = pd.Series(p_gov, index=train.index)
            po_s = pd.Series(p_opp, index=train.index)

            # Recompute spatial lags from OOF predictions.
            train_updated = update_spatial_lags_proba(
                train, pg_s.values, po_s.values, W_cache=W_cache
            )

            new_spat = train_updated[["spat_gov", "spat_opp"]].fillna(0).values
            if prev_spat is not None:
                delta = float(np.abs(new_spat - prev_spat).mean())
                print(f"   FP iter {fp_iter}: Δ spat = {delta:.5f}", end="")
                if delta < FP_TOL:
                    print("  ✓ converged")
                    break
                print()
            else:
                print(f"   FP iter {fp_iter}: initial")

            prev_spat = new_spat
            train = train_updated

        # ── Save outputs ───────────────────────────────────────────────────
        oof_df = train_base[["ddyear", "ccode_A", "ccode_B", "year", "intervention"]].copy()
        oof_df["p_none"] = result["oof_proba"][:, 0]
        oof_df["p_gov"]  = result["oof_proba"][:, 1]
        oof_df["p_opp"]  = result["oof_proba"][:, 2]
        oof_df["cy"]     = cy
        oof_df["ud"]     = ud
        oof_df.to_parquet(INTERIM / f"sl_oof_{cy}_{ud}.parquet", index=False)

        w_df = pd.DataFrame(
            [{"cy": cy, "ud": ud, "method": k, "weight": v}
             for k, v in result["weights"].items()]
        )
        w_df.to_parquet(INTERIM / f"sl_weights_{cy}_{ud}.parquet", index=False)

        # Save model with the final (converged) spatial lags baked in.
        # Notebook 06 will use update_spatial_lags_proba to iterate predictions.
        joblib.dump(result, INTERIM / f"sl_model_{cy}_{ud}.pkl")

        m = result["component_metrics"].copy()
        m["cy"] = cy
        m["ud"] = ud
        all_metrics.append(m)

        sl_row = result["component_metrics"].loc[
            result["component_metrics"]["method"] == "super_learner"
        ]
        print(f"   n_PCA={result['n_components_pca']}  "
              f"SL log-loss={sl_row['log_loss'].values[0]:.4f}  "
              f"SL PRL={sl_row['prl'].values[0]:.3f}")

metrics_all = pd.concat(all_metrics, ignore_index=True)
metrics_all.to_parquet(INTERIM / "sl_cv_metrics.parquet", index=False)

print(f"\nDone. {len(all_metrics)} imputation × model combinations.")


## § 4  Validation & summary table

In [ ]:
summary = (
    metrics_all
    .groupby("method")[["log_loss", "auc", "prl", "weight"]]
    .mean()
    .round(3)
    .sort_values("log_loss")
    .reset_index()
)

# Rename for display
name_map = {
    "rf":           "Random forest",
    "glmnet":       "Elastic net",
    "multinom":     "Multinomial logit",
    "mlp":          "Multilayer perceptron",
    "super_learner": "Super learner",
}
summary["method"] = summary["method"].map(name_map).fillna(summary["method"])

print(summary.to_string(index=False))

# Validation targets (from old paper with R ensemble):
#   Super learner PRL ≈ 0.40+   (vs. null model)
#   Multinomial logit (on full data) PRL ≈ -0.18 (worse than null)
sl_prl = summary.loc[summary["method"] == "Super learner", "prl"].values
if len(sl_prl) > 0:
    assert sl_prl[0] > 0, "Super learner should beat null model"
    print(f"\n✓ Super learner PRL = {sl_prl[0]:.3f} (> 0)")

print(f"\nOOF files produced: {len(list(INTERIM.glob('sl_oof_*.parquet')))}  (expected 25)")